In [1]:
import torch
import torch.nn.functional as F

# Data Preparation

In [2]:
names = []
with open("./data/names.txt", "r") as f:
    for line in f:
        names.append(line.rstrip())

print(len(names))
names[:5]

32033


['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
def create_bigrams(words):
    bigrams = []

    for word in words:
        for c1, c2 in zip(word, word[1:]):
            bigrams.append(c1 + c2)

    return bigrams 

create_bigrams(["rahul"])

['ra', 'ah', 'hu', 'ul']

In [4]:
def pad_words(words):
    padded = []

    for word in words:
        padded.append("".join([".", word, "."]))

    return padded

pad_words(["rahul"])

['.rahul.']

In [5]:
alphabet = ['.'] + sorted(set("".join(names)))
alphabet_len = len(alphabet)

print(alphabet)
print(alphabet_len)

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
27


In [6]:
char_to_idx = {}
for idx, char in enumerate(alphabet):
    char_to_idx[char] = idx

char_to_idx

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [7]:
padded_names = pad_words(names)
padded_names[:5]

['.emma.', '.olivia.', '.ava.', '.isabella.', '.sophia.']

In [8]:
bigrams = create_bigrams(padded_names)
bigrams[:12]

['.e', 'em', 'mm', 'ma', 'a.', '.o', 'ol', 'li', 'iv', 'vi', 'ia', 'a.']

In [9]:
xs = []
ys = []
for bigram in bigrams:
    xs.append(char_to_idx[bigram[0]])
    ys.append(char_to_idx[bigram[1]])

xs[:5]

[0, 5, 13, 13, 1]

# Counting model

In [10]:
generator = torch.Generator().manual_seed(2774943997)

In [11]:
counts = torch.zeros(alphabet_len, alphabet_len)
counts.shape

torch.Size([27, 27])

In [12]:
for x, y in zip(xs, ys):
    counts[x][y] += 1

counts[0][5]

tensor(1531.)

In [13]:
def normalize(tensor):
    row_sums = torch.sum(tensor, dim=1, keepdim=True)

    probs = tensor / row_sums
    assert torch.allclose(probs.sum(dim=1), torch.ones(len(tensor)))

    return probs

In [14]:
probs = normalize(counts + 1)

In [15]:
probs.shape

torch.Size([27, 27])

In [16]:
def generate_names(num, probs_matrix, start_idx):
    words = []
    for _ in range(num):
        word = ""
        next_idx = start_idx
        while True:
            vector = probs_matrix[next_idx]
            idx = torch.multinomial(vector, num_samples=1, replacement=True, generator=generator).item()
            char = alphabet[idx]
            if char == ".":
                break
      
            word += char
            next_idx = idx

        words.append(word)

    return words

generate_names(5, probs, char_to_idx['.'])

['kst', 'allallle', 'susharyn', 'lorist', 'stis']

In [17]:
loss = 0.0
for x, y in zip(xs, ys):
    prob = probs[x][y]
    loss += torch.log(prob)

loss = -(loss / len(bigrams))
loss

tensor(2.4544)

# Neural Network implementation

In [18]:
weights = torch.randn(alphabet_len, alphabet_len, requires_grad=True, generator=generator)

In [19]:
x_oh = F.one_hot(torch.tensor(xs), num_classes=alphabet_len).float()
x_oh.shape

torch.Size([228146, 27])

In [20]:
lr = 65
num_epochs = 100

for epoch in range(num_epochs):
    weights.grad = None

    logits = x_oh @ weights
    nn_probs = normalize(logits.exp())

    loss = -torch.log(nn_probs[torch.arange(len(ys)), ys]).mean()
    loss.backward()

    print(f"{epoch=}: loss={loss.item():4f}")

    with torch.no_grad():
        weights -= lr * weights.grad


epoch=0: loss=3.650144
epoch=1: loss=3.183419
epoch=2: loss=2.997174
epoch=3: loss=2.883796
epoch=4: loss=2.803195
epoch=5: loss=2.744127
epoch=6: loss=2.700292
epoch=7: loss=2.667113
epoch=8: loss=2.641330
epoch=9: loss=2.620827
epoch=10: loss=2.604244
epoch=11: loss=2.590631
epoch=12: loss=2.579296
epoch=13: loss=2.569724
epoch=14: loss=2.561536
epoch=15: loss=2.554449
epoch=16: loss=2.548248
epoch=17: loss=2.542769
epoch=18: loss=2.537888
epoch=19: loss=2.533507
epoch=20: loss=2.529550
epoch=21: loss=2.525958
epoch=22: loss=2.522681
epoch=23: loss=2.519680
epoch=24: loss=2.516921
epoch=25: loss=2.514375
epoch=26: loss=2.512020
epoch=27: loss=2.509835
epoch=28: loss=2.507802
epoch=29: loss=2.505906
epoch=30: loss=2.504133
epoch=31: loss=2.502472
epoch=32: loss=2.500912
epoch=33: loss=2.499445
epoch=34: loss=2.498062
epoch=35: loss=2.496756
epoch=36: loss=2.495521
epoch=37: loss=2.494350
epoch=38: loss=2.493241
epoch=39: loss=2.492186
epoch=40: loss=2.491183
epoch=41: loss=2.490227
ep

In [23]:
generate_names(5, nn_probs, char_to_idx['.'])

['biar', 'waljsykllfr', 'kheiyafszanejtejnieceja', 'dlmraiiygia', 'ana']